In [1]:
import requests as rq
import sqlite3 as sql3
from dotenv import load_dotenv
from random import randint
import os

load_dotenv()

NASA_API=os.getenv('NASA_API')
URL = "https://api.nasa.gov/planetary/apod?api_key=" + str(NASA_API)

response = rq.get(URL)


In [40]:
if response.status_code == 200:
    img_data=response.json()
    print("Request successful")
else:
    print("An error occurred: ", response.status_code)
    exit(1)
    

Request successful


In [56]:
ALLOWED_TABLES = {"apod_images", "words_dict"}

def drop_table(table_name:str, conn) -> None:
    try:
        if table_name not in ALLOWED_TABLES:
            raise ValueError(f"Unknown table: '{table_name}'")

        with sql3.connect('apod_data.db') as conn:
            conn.execute(f"DROP TABLE {table_name};")
            print(f"Table {table_name} dropped.")
        
    except Exception as e:
        print("An error occurred: ", e)

def print_table(table_name:str, conn, limit:int=10) -> None:
    try:
        if table_name not in ALLOWED_TABLES:
            raise ValueError(f"Unknown table: '{table_name}'")

        with sql3.connect('apod_data.db') as conn:
            rows = conn.execute(f"SELECT * FROM {table_name}").fetchmany(limit)
            print(f"Table {table_name}:")
            for row in rows:
                print(row)
            
    except Exception as e:
        print("An error occurred: ", e)

def download_img(image_data:dict):
    image_url = image_data['url']
    date = image_data['date']
    
    try:
        os.makedirs('images/', exist_ok=True)
    
        filename = f"apod_{date}.jpg"
        filepath = os.path.join('images/', filename)

        try:
            img_response = rq.get(image_url)
        except Exception as e:
            print("Image not available, due to error: ", e)
            exit(1)
    
        if img_response.status_code == 200:
            with open(filepath, 'wb') as f:
                f.write(img_response.content)
        
            print(f"image saved: {filepath}")
        else:
            print("An error occurred, the image was not saved, \
            status code:", img_response.status_code)
    
    except Exception as e:
        print("Failed to download image, error: ", e)

def generate_word():
    max_attempts = 20
    
    for attempt in range(max_attempts):
        
        word_response = rq.get("https://random-words-api.kushcreates.com/api?language=en")
        
        if word_response.status_code != 200:
            break
            
        try:
            word_data = word_response.json()
            n = randint(0, len(word_data)-1)
            
            random_word = word_data[n]['word']
            
            dict_response = rq.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{random_word}")
            dict_data = dict_response.json()
            
            if isinstance(dict_data, list):
                print(f"Found valid word: {random_word}")
                return dict_data
            else:
                print(f"Word '{random_word}' not in dictionary, trying again...")
                continue
                
        except Exception as e:
            print("Error:", e)
            continue

    # If the word is default, something went wrong with the word generation
    return 'default'


In [32]:
img_otd = { 
    "title": img_data.get('title', 'No title'),
    "date": img_data.get('date', '01-01-0001'),
    "exp": img_data.get('explanation', 'No explanation'),
    "img_url": img_data.get('url', 'No url'),
    "copyr": img_data.get('copyright', 'No copyright')
}

In [ ]:
with sql3.connect('apod_data.db') as conn:
    try:
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS apod_images (
                id INTEGER PRIMARY KEY,
                title TEXT UNIQUE NOT NULL,
                date TEXT UNIQUE NOT NULL,
                explanation TEXT NOT NULL,
                url TEXT NOT NULL,
                copyright TEXT
            )
        ''')

        cursor.execute('''
            INSERT OR IGNORE INTO apod_images 
                (title, date, explanation, url, copyright)
            VALUES 
                (?, ?, ?, ?, ?)
            ''',(img_otd['title'], img_otd['date'], 
                 img_otd['exp'], img_otd['img_url'], 
                 img_otd['copyr']))

    except Exception as e:
        print("Error: ", e)


In [ ]:
download_img(img_data)

image saved: images/apod_2026-03-06.jpg


In [47]:
dict_data = generate_word()
    
if dict_data=='default':
    word_otd = {
        "word": 'default',
        "definition": 'automatic or standard way of acting or responding.'
    }
else:
    word_otd = {
        "word": dict_data[0].get("word", 'No word'),
        "definition": dict_data[0]['meanings'][0]['definitions'][0].get("definition", 'No definition')
    }

Word 'fauts' not in dictionary, trying again...
Found valid word: amour


In [49]:
with sql3.connect('apod_data.db') as conn:
    try:
        cursor = conn.cursor()
        cursor.execute('''
        CREATE TABLE IF NOT EXISTS words_dict (
            id INTEGER PRIMARY KEY,
            word TEXT UNIQUE NOT NULL,
            definition TEXT NOT NULL
            )
        ''')

        cursor.execute('''
            INSERT OR IGNORE INTO words_dict 
            (word, definition)
            VALUES 
            (?, ?)
            ''', (word_otd['word'], word_otd['definition']))
    
    except Exception as e:
        print("Error: ", e)
    

In [53]:
print_table("apod_images", conn)
print("\n")
print_table("words_dict", conn)

An error occurred:  no such table: apod_images


An error occurred:  no such table: words_dict
